***

Preparing Workspace

***

In [ ]:


## Importing packages ----

import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
# pd.options.display.float_format = '{:.0f}'.format


## Setting file paths ---

# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

    # SharePoint
    path_out  = os.path.join(path_users
                             , 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents'
                             , 'Process Revamp'
                             , 'Task 9. Collect new data'
                             , 'Census')

path_code    = os.path.join(path_git, 'Data', 'Census')
path_config0 = os.path.join(path_git , 'config')
path_config  = os.path.join(path_code, 'config')


## User defined functions ---

exec(open(os.path.join(path_config0,        'Functions.py')).read())
exec(open(os.path.join(path_config , 'Census Functions.py')).read())


## Setting API key ---

# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()



***

Preparing Import Parameters

***

In [ ]:


# Execute script to prepare API request inputs
exec(open(os.path.join(path_code, 'Supplemental Scripts', 'Step 01a - Prepare API Request Inputs.py')).read())



***

Importing

***

In [ ]:


# Execute script to import Census Bureau Data
exec(open(os.path.join(path_code, 'Supplemental Scripts', 'Step 01b - Run API Queries.py')).read())



***

Exporting

***

In [ ]:


# If PUMA, change estimate title
if geography == 'PUMA':
    estimate = re.sub('ACS', 'PUMS', estimate)

# Name of the export
if margin_of_error == 'No':
    end = 'NoME_raw.csv'
else:
    end = 'raw.csv'
export_title = f"{indicator_name}_{geography}_{estimate}_{end}"


# Export location
print(f"Exporting {export_title} to the following location: ")
print(path_out)


# Export
df_census_raw.to_csv(os.path.join(path_out, export_title), index = False)

print('')
print('Successfully exported!')



***

Processing (optional)

***

In [ ]:


df_census = df_census_raw.copy()

if sample_type in ['ACS', 'SUBJECT']:
    df_census = acs_processing_1(df_census, df_vars, geography, margin_of_error)
    display(df_census.head(3), df_census.tail(3))

if sample_type in ['PUMS', 'FOODSEC']:
    df_census, groups = pums_processing_1(df_census, df_vars, sample_type, weight)
    print('Groups: ' + ', '.join(groups))
    display(df_census.head(3), df_census.tail(3))

## LEHD processing steps are still a work in progress
percentages = 'Yes'
if estimate == 'LEHD':
    if geography == 'Counties':
        df_counties, df_mpo = lehd_processing(df_census, geography, indicator_name, percentages, df_fips)
        display(df_counties.head(3), df_mpo.head(3))
    if geography == 'MSA':
        df_msa = lehd_processing(df_census, geography, indicator_name, percentages)
        display(df_msa.head(3))

